# Get monthly data from Ed's Rainfall Rescue project

A key part of processing the daily rainfall data is to match it to the monthly rainfall data already digitized by Rainfall Rescue. The monthly data already has station locations and other metadata, so if we can match a daily record to a monthly one, we can use the metadata from the monthly data to enrich the daily data.

Get Ed's RR data from GitHub, and ingest it into an SQLite database for easy processing.

Run this notebook with the ADRQ kernel.

In [3]:
# Setup

import os
import subprocess

# Target location for the Rainfall Rescue repository
repo_dir = f"{os.getenv('PDIR')}/Rainfall-Rescue"

# Repository URL for Rainfall Rescue
repo_url = "git@github.com:ed-hawkins/rainfall-rescue.git"

In [ ]:
# Clone the Rainfall Rescue GitHub repository if it doesn't already exist on local disc

if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", repo_url, repo_dir])

In [4]:
# Set up input and output paths for the RR data ingest

from pathlib import Path

from src.rainfall_rescue_sqlite.ingest import default_db_path, ingest_combined_csvs

rainfall_rescue_root = Path(repo_dir)
db_path = default_db_path()

In [ ]:
# Do the data ingestion (note, takes 20 minutes). 
# Performs a full rebuild of the SQLite database from combined files in `DATA` (excluding `TYRain_*.csv` source sheets).

result = ingest_combined_csvs(
    rainfall_rescue_root=rainfall_rescue_root,
    db_path=db_path,
)

result

In [4]:
# Test import - show station counts and sample data

import sqlite3

with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()
    station_count = cur.execute("SELECT COUNT(*) FROM stations").fetchone()[0]
    monthly_count = cur.execute("SELECT COUNT(*) FROM monthly_rainfall").fetchone()[0]
    annual_count = cur.execute("SELECT COUNT(*) FROM annual_totals").fetchone()[0]

    print(f"Stations: {station_count}")
    print(f"Monthly rows: {monthly_count}")
    print(f"Annual rows: {annual_count}")

    sample = cur.execute(
        """
        SELECT s.location_name, m.year, m.month, m.rainfall_in
        FROM monthly_rainfall m
        JOIN stations s ON s.station_file_id = m.station_file_id
        WHERE s.location_name LIKE 'ABERPORTH%'
        ORDER BY m.year, m.month
        LIMIT 24
        """
    ).fetchall()

sample[:5]

Stations: 8690
Monthly rows: 3374861
Annual rows: 277250


[('ABERPORTH', 1941, 1, 2.94),
 ('ABERPORTH', 1941, 2, 2.72),
 ('ABERPORTH', 1941, 3, 3.0),
 ('ABERPORTH', 1941, 4, 1.33),
 ('ABERPORTH', 1941, 5, 2.02)]

In [5]:
# Retrieve RR station metadata for a few stations

with sqlite3.connect(db_path) as conn:
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    stations = cur.execute(
        """
        SELECT
            station_file_id,
            location_name,
            station_number,
            grid_reference,
            longitude,
            latitude,
            elevation_ft,
            station_file_name,
            source_path
        FROM stations
        ORDER BY location_name
        LIMIT 5
        """
    ).fetchall()

for row in stations:
    print(
        f"{row['location_name']!s:<40}"
        f"  stn={row['station_number'] or 'n/a':<12}"
        f"  lat={row['latitude'] or 'n/a'!s:<10}"
        f"  lon={row['longitude'] or 'n/a'!s:<12}"
        f"  elev={row['elevation_ft'] or 'n/a'!s:<6} ft"
        f"  grid={row['grid_reference'] or 'n/a'}"
    )

1 MI FROM EDINBURGH                       stn=RR502         lat=55.94251    lon=-3.187925     elev=250    ft  grid=NT259728
63 BLOOMSBURY ST BIRMINGHAM               stn=RR1550        lat=52.4922     lon=-1.8731       elev=340    ft  grid=SP087882
ABBEY ST BATHANS, BERWICKSHIRE            stn=RR2226        lat=55.852382   lon=-2.3897074    elev=450    ft  grid=NT757622
ABBEY-CWMHIR-THE-HALL                     stn=4241/5        lat=52.330722   lon=-3.3881959    elev=877    ft  grid=SO055712
ABBEY-LEIX-BLANDSFORT                     stn=n/a           lat=52.930139   lon=-7.284667     elev=532    ft  grid=IS4814786802


## Interactive map of monthly Rainfall Rescue data

Plot the locations of all Rainfall Rescue stations for a chosen year and month.
Stations with a rainfall value for that month are coloured by the amount (inches);
stations with no value for that month are drawn as pale grey circles underneath.
Hover over a point to see the station name, number and value.

In [5]:
# Interactive map of monthly RR rainfall for a chosen year and month.

import calendar
import sqlite3

import plotly.graph_objects as go


def build_rr_monthly_figure(year, month, db_path, *, cmap="YlGnBu", marker_size=8.0):
    """Plot all located RR stations, colouring those with data for (year, month)."""
    with sqlite3.connect(f"file:{db_path}?immutable=1", uri=True) as conn:
        conn.row_factory = sqlite3.Row
        cur = conn.cursor()

        located = cur.execute(
            """
            SELECT station_file_id, location_name, station_number, latitude, longitude
            FROM stations
            WHERE latitude IS NOT NULL AND longitude IS NOT NULL
            """
        ).fetchall()

        values = {
            r["station_file_id"]: r["rainfall_in"]
            for r in cur.execute(
                """
                SELECT station_file_id, rainfall_in
                FROM monthly_rainfall
                WHERE year = ? AND month = ? AND rainfall_in IS NOT NULL
                """,
                (year, month),
            ).fetchall()
        }

    valued = [r for r in located if r["station_file_id"] in values]
    null = [r for r in located if r["station_file_id"] not in values]

    vals = [values[r["station_file_id"]] for r in valued]
    vmax = 1.0
    if vals:
        ordered = sorted(vals)
        idx = min(len(ordered) - 1, int(round(0.98 * (len(ordered) - 1))))
        vmax = max(ordered[idx] or max(vals), 1e-6)

    fig = go.Figure()

    # Null (no value this month) stations first, so coloured points draw on top.
    if null:
        fig.add_trace(
            go.Scattergeo(
                lon=[r["longitude"] for r in null],
                lat=[r["latitude"] for r in null],
                mode="markers",
                name="no value",
                marker=dict(
                    size=marker_size,
                    color="lightgrey",
                    line=dict(width=0.5, color="grey"),
                ),
                customdata=[
                    [r["location_name"] or "", r["station_number"] or "n/a"]
                    for r in null
                ],
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "station %{customdata[1]}<br>"
                    "no value for this month"
                    "<extra></extra>"
                ),
            )
        )

    if valued:
        fig.add_trace(
            go.Scattergeo(
                lon=[r["longitude"] for r in valued],
                lat=[r["latitude"] for r in valued],
                mode="markers",
                name="monthly rainfall",
                marker=dict(
                    size=marker_size + 1,
                    color=vals,
                    colorscale=cmap,
                    cmin=0.0,
                    cmax=vmax,
                    line=dict(width=0.5, color="black"),
                    colorbar=dict(title="Monthly<br>rainfall (in)"),
                ),
                customdata=[
                    [
                        r["location_name"] or "",
                        r["station_number"] or "n/a",
                        values[r["station_file_id"]],
                    ]
                    for r in valued
                ],
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "station %{customdata[1]}<br>"
                    "rainfall: %{customdata[2]:.2f} in"
                    "<extra></extra>"
                ),
            )
        )

    fig.update_geos(
        resolution=50,
        scope="europe",
        showcountries=True,
        countrycolor="black",
        showland=True,
        landcolor="rgb(243, 243, 243)",
        showocean=True,
        oceancolor="rgb(230, 240, 250)",
        lataxis_range=[49, 61],
        lonaxis_range=[-11, 4],
    )
    fig.update_layout(
        title=(
            f"Rainfall Rescue monthly rainfall  {calendar.month_name[month]} {year}<br>"
            f"<sup>{len(valued)} of {len(located)} located stations have data "
            f"for this month</sup>"
        ),
        width=800,
        height=1000,
        margin=dict(l=10, r=10, t=70, b=10),
    )
    return fig


# Change year/month to any period covered by the RR data.
build_rr_monthly_figure(year=1931, month=10, db_path=db_path)